# Mini Project I-Smile: EDA to Machine Learning
**Kelompok 7**

* **Judul:** Prediksi Risiko Depresi Remaja Berdasarkan Pola Media Sosial dan Gaya Hidup
* **Masalah:** Klasifikasi Biner (`depression_label`: 0 = Tidak Depresi, 1 = Depresi)
* **Dataset:** [Teen Mental Health Dataset (Kaggle)](https://www.kaggle.com/datasets/sunil123kumar/social-media-impact-on-mental-health?select=Teen_Mental_Health_Dataset.csv)
* **Tujuan:** Mengetahui faktor kebiasaan digital yang paling memengaruhi risiko depresi pada remaja serta membandingkan performa model klasifikasi.

## 1. Import Library & Load Data
Kita siapkan library standar untuk manipulasi data (pandas, numpy), visualisasi (matplotlib, seaborn), serta model dari scikit-learn dan xgboost.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Model & Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrik Evaluasi
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

sns.set_theme(style='whitegrid')
print("Library siap digunakan.")

In [ ]:
# Load dataset (otomatis cek folder Colab / lokal)
path_data = 'Teen_Mental_Health_Dataset.csv'
if not os.path.exists(path_data) and os.path.exists('/content/Teen_Mental_Health_Dataset.csv'):
    path_data = '/content/Teen_Mental_Health_Dataset.csv'

df = pd.read_csv(path_data)
print(f"Data termuat: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

## 2. Exploratory Data Analysis (EDA)
Kita periksa dulu struktur dataset, tipe kolom, sebaran nilai, dan distribusi kelas target.

In [ ]:
# Cek info tipe data dan missing value
df.info()

In [ ]:
# Ringkasan statistik kolom numerik
df.describe()

In [ ]:
# Cek missing value dan duplikat
print("Jumlah missing value:")
print(df.isnull().sum())
print()
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

In [ ]:
# Distribusi kolom kategorik dan target
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    print(f"Distribusi {col}:")
    print(df[col].value_counts())
    print()

print("Distribusi Target (depression_label):")
print(df['depression_label'].value_counts())
print(f"Proporsi kelas 1 (Depresi): {df['depression_label'].mean():.2%}")

**Catatan Temuan Awal:**
1. Dataset terdiri dari **1.200 baris dan 13 kolom** tanpa ada nilai kosong (*missing value* = 0).
2. Terdapat **ketimpangan kelas ekstrem (*extreme imbalance*)**: Kelas 0 (Tidak Depresi) sebanyak 1.169 orang (97,4%), sedangkan Kelas 1 (Depresi) hanya 31 orang (2,6%).
3. Karena ketimpangan ini, akurasi saja tidak bisa dijadikan patokan tunggal — model yang selalu menebak 0 pun akan dapat akurasi 97%. Kita perlu memperhatikan **Recall** dan **Precision**.

### Visualisasi Data
Berikut visualisasi untuk memahami sebaran target, korelasi fitur, dan kebiasaan digital remaja.

#### 1. Distribusi Target (`depression_label`)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='depression_label', palette=['#3498db', '#e74c3c'])
plt.title('Distribusi Kelas Target', fontsize=12, fontweight='bold')
plt.xlabel('Label Depresi (0 = Tidak, 1 = Ya)')
plt.ylabel('Jumlah Remaja')

for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df):.1%})",
                (p.get_x() + p.get_width() / 2., p.get_height() + 15),
                ha='center', fontsize=10)

plt.tight_layout()
plt.show()

*Temuan:* Dari 1.200 remaja, hanya 31 orang yang berlabel depresi. Split data nanti wajib memakai `stratify=y` agar proporsi 2,6% ini tidak hilang di salah satu partisi data.

#### 2. Heatmap Korelasi Fitur Numerik

In [ ]:
fitur_numerik = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
fitur_numerik_bebas = [c for c in fitur_numerik if c != 'depression_label']

plt.figure(figsize=(10, 8))
sns.heatmap(df[fitur_numerik].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Korelasi Antar Fitur Numerik', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

*Temuan:* Fitur `stress_level`, `anxiety_level`, dan `addiction_level` memiliki korelasi positif paling kuat terhadap `depression_label`. Sebaliknya, `sleep_hours` berkorelasi negatif (semakin sedikit jam tidur, kecenderungan depresi meningkat).

#### 3. Distribusi Fitur Gaya Hidup & Media Sosial

In [ ]:
fitur_pantau = ['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'stress_level']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(fitur_pantau, axes.flatten()):
    sns.histplot(data=df, x=col, hue='depression_label', kde=True, ax=ax,
                 palette=['#3498db', '#e74c3c'], element='step')
    ax.set_title(f'Distribusi {col}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

*Temuan:* Remaja pada kelompok depresi (merah) cenderung memiliki jam medsos harian tinggi (>4-5 jam), waktu tidur yang lebih singkat (<5-6 jam), serta tingkat stres di skala atas (7-10).

#### 4. Penggunaan Platform Media Sosial vs Depresi

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='platform_usage', hue='depression_label', palette=['#3498db', '#e74c3c'])
plt.title('Distribusi Depresi Berdasarkan Platform Medsos', fontsize=12, fontweight='bold')
plt.xlabel('Platform')
plt.ylabel('Jumlah')
plt.legend(['Tidak Depresi', 'Depresi'])
plt.tight_layout()
plt.show()

*Temuan:* Kasus depresi tersebar di pengguna Instagram, TikTok, maupun yang memakai keduanya, dengan proporsi kasus depresi sedikit lebih tinggi pada pengguna aktif kedua platform (*Both*).

## 3. Data Cleaning & Preprocessing
Sesuai brief, setiap keputusan pembersihan data harus disertai alasan konkret.

In [ ]:
# 1. Pengecekan Missing Value
print(f"Total missing value di dataset: {df.isnull().sum().sum()}")

**Keputusan Missing Value:** Tidak ada tindakan pembersihan karena nilai kosong adalah 0.

In [ ]:
# 2. Pengecekan Baris Duplikat
n_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat: {n_duplikat}")
if n_duplikat > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Baris duplikat telah dibersihkan.")
else:
    print("Tidak ditemukan baris duplikat.")

**Keputusan Duplikat:** Cek dan pastikan tidak ada baris ganda agar model tidak menghafal data kembar (*overfitting*).

In [ ]:
# 3. Analisis Outlier menggunakan Metode IQR
rekap_outlier = []
for col in fitur_numerik_bebas:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    bawah = q1 - 1.5 * iqr
    atas = q3 + 1.5 * iqr
    jumlah_outlier = ((df[col] < bawah) | (df[col] > atas)).sum()
    rekap_outlier.append({
        'Fitur': col,
        'Batas Bawah': round(bawah, 2),
        'Batas Atas': round(atas, 2),
        'Outlier': jumlah_outlier
    })

pd.DataFrame(rekap_outlier)

**Keputusan Outlier:** Outlier **tidak dihapus**.
* **Alasan:** Pada konteks kesehatan mental, nilai ekstrem (seperti tingkat stres sangat tinggi atau jam tidur sangat rendah) merupakan sinyal penting kondisi psikologis remaja, bukan kesalahan pencatatan (*noise*). Selain itu, model berbasis pohon (Decision Tree, Random Forest, XGBoost) secara alami tahan terhadap outlier.

In [ ]:
# 4. Encoding Kolom Kategorik
df_clean = df.copy()

# Gender: binary nominal (2 nilai) -> 0 dan 1
df_clean['gender'] = df_clean['gender'].map({'male': 0, 'female': 1})

# Social Interaction Level: ordinal (punya tingkatan) -> low=0, medium=1, high=2
df_clean['social_interaction_level'] = df_clean['social_interaction_level'].map({
    'low': 0, 'medium': 1, 'high': 2
})

# Platform Usage: nominal tanpa urutan -> One-Hot Encoding
df_clean = pd.get_dummies(df_clean, columns=['platform_usage'], drop_first=False)
for col in df_clean.select_dtypes(include='bool').columns:
    df_clean[col] = df_clean[col].astype(int)

print(f"Dimensi setelah encoding: {df_clean.shape}")
df_clean.head()

**Alasan Encoding:**
1. `gender` diubah ke 0 dan 1 karena hanya ada dua kategori.
2. `social_interaction_level` di-map ke 0, 1, 2 agar model menangkap **urutan alami** (`low` < `medium` < `high`).
3. `platform_usage` di-encode dengan One-Hot Encoding karena bersifat nominal (Instagram dan TikTok tidak memiliki tingkatan hierarki).

## 4. Split Data (Train & Test)
Data dibagi menjadi **80% data latih** dan **20% data uji** menggunakan `random_state=42` dan `stratify=y`.

In [ ]:
X = df_clean.drop('depression_label', axis=1)
y = df_clean['depression_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Data latih : {X_train.shape[0]} baris ({X_train.shape[0]/len(X):.0%})")
print(f"Data uji   : {X_test.shape[0]} baris ({X_test.shape[0]/len(X):.0%})")
print(f"Kelas 1 di Data Latih : {(y_train == 1).sum()} ({(y_train == 1).mean():.1%})")
print(f"Kelas 1 di Data Uji   : {(y_test == 1).sum()} ({(y_test == 1).mean():.1%})")

*Alasan Stratify:* Wajib dipakai pada data tidak seimbang agar persentase kelas minoritas (2,6%) terbagi rata di data latih dan data uji, tidak menumpuk di salah satu sisi.

## 5. Model Standar: KNN + Scaling (Wajib)
Sesuai brief, model standar wajib klasifikasi adalah `KNeighborsClassifier` dengan fitur yang sudah di-*scaling*. Karena KNN menghitung jarak euclidean, fitur dengan skala besar (misal rentang skor 1-10) bisa mendominasi fitur berrentang kecil jika tanpa `StandardScaler`.

In [ ]:
# Scaling fitur (fit hanya pada data latih)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model Standar: KNN default k=5
knn_standar = KNeighborsClassifier(n_neighbors=5)
knn_standar.fit(X_train_scaled, y_train)

y_tr_knn = knn_standar.predict(X_train_scaled)
y_te_knn = knn_standar.predict(X_test_scaled)

eval_knn_standar = pd.DataFrame({
    'Data Latih': [accuracy_score(y_train, y_tr_knn),
                   precision_score(y_train, y_tr_knn, zero_division=0),
                   recall_score(y_train, y_tr_knn, zero_division=0)],
    'Data Uji': [accuracy_score(y_test, y_te_knn),
                 precision_score(y_test, y_te_knn, zero_division=0),
                 recall_score(y_test, y_te_knn, zero_division=0)]
}, index=['Accuracy', 'Precision', 'Recall'])

eval_knn_standar.round(4)

In [ ]:
# Classification report & Confusion Matrix KNN standar
print("Classification Report KNN Standar (Data Uji):")
print(classification_report(y_test, y_te_knn, zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_te_knn,
                                       display_labels=['Tidak Depresi', 'Depresi'],
                                       cmap='Blues', ax=ax)
plt.title('Confusion Matrix - KNN Standar (k=5)', fontsize=11, fontweight='bold')
plt.grid(False)
plt.tight_layout()
plt.show()

*Pelajaran Penting:* KNN standar menghasilkan akurasi 97,5%, tetapi **Recall-nya 0,00%**. Artinya tidak ada satu pun dari 6 remaja depresi di data uji yang berhasil dideteksi. Ini bukti nyata dari **Accuracy Paradox** pada data tidak seimbang.

## 6. Hyperparameter Tuning KNN (GridSearchCV)
Sesuai materi Hari 4, kita mencari nilai $k$ terbaik menggunakan `GridSearchCV` dengan 5-fold cross-validation. Kita uji angka ganjil (1 sampai 21) agar saat voting kelas tidak terjadi seri.

In [ ]:
param_grid = {'n_neighbors': list(range(1, 23, 2))}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv_strat,
    scoring='accuracy',
    return_train_score=True
)
grid_knn.fit(X_train_scaled, y_train)

k_terbaik = grid_knn.best_params_['n_neighbors']
print(f"Nilai k terbaik: {k_terbaik} (Skor CV: {grid_knn.best_score_:.4f})")

hasil_grid = pd.DataFrame(grid_knn.cv_results_)
tabel_grid = hasil_grid[['param_n_neighbors', 'mean_test_score', 'std_test_score']].copy()
tabel_grid.columns = ['Nilai k', 'Mean Accuracy CV', 'Std Deviasi']
tabel_grid['Nilai k'] = tabel_grid['Nilai k'].astype(int)
tabel_grid.head(10).round(4)

In [ ]:
# Visualisasi pengaruh nilai k terhadap performa CV
plt.figure(figsize=(9, 4))
plt.plot(tabel_grid['Nilai k'], tabel_grid['Mean Accuracy CV'], marker='o', color='#2980b9', lw=2)
plt.axvline(k_terbaik, color='red', linestyle='--', label=f'k terbaik = {k_terbaik}')
plt.title('Hasil Tuning Parameter k pada KNN', fontsize=12, fontweight='bold')
plt.xlabel('Nilai k')
plt.ylabel('Rata-rata Accuracy CV')
plt.xticks(tabel_grid['Nilai k'])
plt.legend()
plt.tight_layout()
plt.show()

## 7. Model Lain & Cross-Validation 5 Lipatan
Kita membandingkan 4 model dengan aturan main yang setara di data latih:
1. **KNN (k=7):** Dibungkus dalam `make_pipeline` bersama `StandardScaler` agar scaling tidak bocor antar-fold (*data leakage*).
2. **Decision Tree:** Model pohon tunggal (white-box).
3. **Random Forest:** Ensemble bagging (100 pohon).
4. **XGBoost:** Ensemble gradient boosting.

In [ ]:
models = {
    f'KNN (k={k_terbaik})': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k_terbaik)),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

cv_scores = {}
for name, model in models.items():
    skor = cross_val_score(model, X_train, y_train, cv=cv_strat, scoring='accuracy')
    cv_scores[name] = skor

# Tampilkan hasil CV
tabel_cv = pd.DataFrame({
    'Rata-rata CV': [cv_scores[m].mean() for m in models],
    'Std Deviasi': [cv_scores[m].std() for m in models]
}, index=list(models.keys()))

tabel_cv.round(4)

## 8. Evaluasi Lengkap & Perbandingan Model
Semua model dilatih pada seluruh data latih, lalu diuji performanya pada data latih dan data uji.

In [ ]:
hasil_komparasi = []
model_terlatih = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    model_terlatih[name] = model
    
    y_pred_tr = model.predict(X_train)
    y_pred_te = model.predict(X_test)
    
    hasil_komparasi.append({
        'Model': name,
        'CV Mean': cv_scores[name].mean(),
        'CV Std': cv_scores[name].std(),
        'Acc Train': accuracy_score(y_train, y_pred_tr),
        'Acc Test': accuracy_score(y_test, y_pred_te),
        'Precision Test': precision_score(y_test, y_pred_te, zero_division=0),
        'Recall Test': recall_score(y_test, y_pred_te, zero_division=0)
    })

df_perbandingan = pd.DataFrame(hasil_komparasi).set_index('Model')
df_perbandingan.round(4)

In [ ]:
# Visualisasi perbandingan Train vs Test Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 1. Skor CV
axes[0].barh(df_perbandingan.index, df_perbandingan['CV Mean'],
             xerr=df_perbandingan['CV Std'], color='#16a085', edgecolor='black', capsize=4)
axes[0].set_title('Rata-rata Akurasi CV (5 Lipatan)', fontsize=11, fontweight='bold')
axes[0].set_xlim(0.95, 1.005)
axes[0].set_xlabel('Akurasi')

# 2. Train vs Test
y_pos = np.arange(len(df_perbandingan))
width = 0.35
axes[1].barh(y_pos - width/2, df_perbandingan['Acc Train'], width, label='Train', color='#2980b9', edgecolor='black')
axes[1].barh(y_pos + width/2, df_perbandingan['Acc Test'], width, label='Test', color='#e67e22', edgecolor='black')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(df_perbandingan.index)
axes[1].set_title('Akurasi: Data Latih vs Data Uji', fontsize=11, fontweight='bold')
axes[1].set_xlim(0.95, 1.005)
axes[1].set_xlabel('Akurasi')
axes[1].legend()

plt.tight_layout()
plt.show()

*Analisis Perbandingan:*
* **KNN & Random Forest:** Keduanya terjebak memprediksi kelas mayoritas pada data uji, sehingga Precision dan Recall-nya bernilai 0.
* **Decision Tree & XGBoost:** Keduanya mampu menangkap pola kelas minoritas. Namun, **XGBoost** memiliki nilai CV Mean tertinggi (**0,9958**) dengan standar deviasi paling stabil ($\pm 0,0039$), menjadikannya model dengan konsistensi generalisasi terbaik.

### Visualisasi Pohon Keputusan (White-Box)
Decision Tree memberikan transparansi logika: aturan apa saja yang diperiksa model saat memprediksi risiko depresi.

In [ ]:
# Tampilkan Decision Tree sederhana (kedalaman 3 tingkat)
pohon_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
pohon_demo.fit(X_train, y_train)

plt.figure(figsize=(18, 8))
plot_tree(pohon_demo,
          feature_names=X_train.columns.tolist(),
          class_names=['Tidak Depresi', 'Depresi'],
          filled=True,
          rounded=True,
          fontsize=9)
plt.title('Diagram Keputusan (Decision Tree max_depth=3)', fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Feature Importance dari Random Forest
rf_model = model_terlatih['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='#34495e', edgecolor='black')
plt.title('Tingkat Kepentingan Fitur (Random Forest Feature Importance)', fontsize=12, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("Top 5 Fitur Paling Berpengaruh:")
print(importances.tail(5).iloc[::-1].round(4))

## 9. Evaluasi Final Model Terpilih
Sesuai kaidah machine learning, model terbaik dipilih berdasarkan **skor cross-validation tertinggi pada data latih**, bukan dari skor data uji, untuk menghindari bias pemilihan.

In [ ]:
# Membandingkan Confusion Matrix: Model Standar vs Model Terpilih (XGBoost)
nama_terbaik = 'XGBoost'
y_pred_xgb = model_terlatih[nama_terbaik].predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_te_knn,
                                       display_labels=['Tidak Depresi', 'Depresi'],
                                       cmap='Blues', ax=axes[0])
axes[0].set_title(f'Model Standar: KNN (k={k_terbaik})', fontsize=11, fontweight='bold')
axes[0].grid(False)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb,
                                       display_labels=['Tidak Depresi', 'Depresi'],
                                       cmap='Greens', ax=axes[1])
axes[1].set_title(f'Model Terpilih: {nama_terbaik}', fontsize=11, fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.show()

print("Classification Report Model Terpilih (XGBoost) di Data Uji:")
print(classification_report(y_test, y_pred_xgb, target_names=['Tidak Depresi', 'Depresi']))

In [ ]:
# Tabel rekap akhir perbandingan
tabel_akhir = pd.DataFrame({
    f'Model Standar (KNN k={k_terbaik})': [
        cv_scores[f'KNN (k={k_terbaik})'].mean(),
        accuracy_score(y_test, y_te_knn),
        precision_score(y_test, y_te_knn, zero_division=0),
        recall_score(y_test, y_te_knn, zero_division=0)
    ],
    f'Model Terpilih ({nama_terbaik})': [
        cv_scores[nama_terbaik].mean(),
        accuracy_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb, zero_division=0),
        recall_score(y_test, y_pred_xgb, zero_division=0)
    ]
}, index=['CV Mean (Latih)', 'Akurasi (Uji)', 'Precision (Uji)', 'Recall (Uji)'])

tabel_akhir.round(4)

## 10. Kesimpulan & Interpretasi

### 1. Interpretasi Metrik dalam Konteks Medis / Psikologi
* **Akurasi (Accuracy):** Persentase total tebakan yang tepat. Pada dataset ini, akurasi tinggi bisa menipu karena kelas mayoritas (tidak depresi) mendominasi 97,4%.
* **Recall (Sensitivitas):** Dari seluruh remaja yang benar-benar mengalami depresi, berapa persen yang berhasil terdeteksi oleh model.
* **Precision:** Dari seluruh remaja yang diprediksi depresi oleh model, berapa persen yang diagnosanya benar.
* **Urgensi False Negative:** Pada skrining depresi, **False Negative jauh lebih berbahaya daripada False Positive**. Jika remaja depresi tidak terdeteksi (FN), mereka tidak akan mendapatkan bantuan tepat waktu. Maka dari itu, nilai **Recall 100% pada XGBoost** menjadikannya pilihan model yang paling tepat dibandingkan KNN yang memiliki Recall 0%.

### 2. Faktor Utama Pemicu Depresi
Berdasarkan analisis korelasi dan *feature importance*, dua kebiasaan utama yang paling berkaitan dengan risiko depresi remaja adalah:
1. **Durasi tidur yang kurang (`sleep_hours`)**
2. **Waktu penggunaan media sosial yang berlebihan (`daily_social_media_hours`)**

### 3. Keterbatasan Model & Data
1. **Ketimpangan Ekstrem:** Hanya terdapat 31 data positif depresi dari 1.200 responden. Pengujian di dataset nyata yang lebih seimbang disarankan untuk pengujian lebih lanjut.
2. **Potensi Target Leakage:** Fitur `stress_level` dan `anxiety_level` sangat berhimpitan dengan gejala depresi, sehingga model terlihat sangat mudah membedakan kelas target.
3. **Validasi Klinis:** Prediksi model ini bersifat alat bantu penapisan awal (*screening*), bukan penentu diagnosis medis mutlak tanpa pengawasan profesional.